# LVS-VAE Dataset Analysis


Use this notebook to inspect the compact NumPy dataset and the CSV/parquet-style dataframes generated by `DATASET_GENERATOR.ipynb`.

### 1. Imports and Paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
DATASET_DIR = PROJECT_ROOT / "VAE_DATA" / "full_20k_40_40_20"

NPZ_PATH = DATASET_DIR / "states_labels.npz"
DATASET_CSV_PATH = DATASET_DIR / "dataset.csv"
METADATA_CSV_PATH = DATASET_DIR / "metadata.csv"
DATASET_PARQUET_PATH = DATASET_DIR / "dataset.parquet"
METADATA_PARQUET_PATH = DATASET_DIR / "metadata.parquet"

DATASET_DIR

WindowsPath('C:/Users/chris/OneDrive/Documents/GitHub/TNG_Falcon/VAE_DATA/full_20k_40_40_20')

### 2. File Availability

In [2]:
files_df = pd.DataFrame(
    [
        {"artifact": "states_labels.npz", "path": NPZ_PATH, "exists": NPZ_PATH.exists()},
        {"artifact": "dataset.csv", "path": DATASET_CSV_PATH, "exists": DATASET_CSV_PATH.exists()},
        {"artifact": "metadata.csv", "path": METADATA_CSV_PATH, "exists": METADATA_CSV_PATH.exists()},
        {"artifact": "dataset.parquet", "path": DATASET_PARQUET_PATH, "exists": DATASET_PARQUET_PATH.exists()},
        {"artifact": "metadata.parquet", "path": METADATA_PARQUET_PATH, "exists": METADATA_PARQUET_PATH.exists()},
    ]
)
files_df["path"] = files_df["path"].astype(str)
files_df

,artifact,path,exists
0,states_labels.npz,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,True
1,dataset.csv,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,True
2,metadata.csv,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,True
3,dataset.parquet,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,False
4,metadata.parquet,C:\Users\chris\OneDrive\Documents\GitHub\TNG_F...,False


### 3. Load NPZ Arrays

In [3]:
with np.load(NPZ_PATH, allow_pickle=False) as npz_data:
    print("NPZ keys:", npz_data.files)
    states = npz_data["states"]
    labels = npz_data["labels"]
    state_columns = npz_data["state_columns"] if "state_columns" in npz_data.files else np.array([])

npz_summary_df = pd.DataFrame(
    [
        {"array": "states", "shape": states.shape, "dtype": states.dtype},
        {"array": "labels", "shape": labels.shape, "dtype": labels.dtype},
        {"array": "state_columns", "shape": state_columns.shape, "dtype": state_columns.dtype},
    ]
)
npz_summary_df

NPZ keys: ['states', 'labels', 'state_columns']


,array,shape,dtype
0,states,"(20000, 25)",int8
1,labels,"(20000,)",float32
2,state_columns,"(25,)",<U17


### 4. View NPZ as DataFrame

In [4]:
state_columns = [str(column) for column in state_columns]
npz_dataset_df = pd.DataFrame(states, columns=state_columns)
npz_dataset_df["value_label"] = labels
npz_dataset_df.head()

,cell_00,cell_01,cell_02,cell_03,cell_04,cell_05,cell_06,cell_07,cell_08,cell_09,...,cell_16,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,goats_eaten_state,phase_state,value_label
0,2,0,0,2,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.0
1,2,0,0,2,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1.0
2,2,0,2,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1.0
3,0,0,2,0,2,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1.0
4,1,2,0,0,2,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1.0


### 5. Load CSV or Parquet DataFrames

In [5]:
def load_table(parquet_path, csv_path):
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f"Missing table: {parquet_path} or {csv_path}")


dataset_df = load_table(DATASET_PARQUET_PATH, DATASET_CSV_PATH)
metadata_df = load_table(METADATA_PARQUET_PATH, METADATA_CSV_PATH)

pd.DataFrame(
    [
        {"dataframe": "dataset_df", "rows": len(dataset_df), "columns": dataset_df.shape[1]},
        {"dataframe": "metadata_df", "rows": len(metadata_df), "columns": metadata_df.shape[1]},
    ]
)

,dataframe,rows,columns
0,dataset_df,20000,26
1,metadata_df,20000,12


### 6. Preview Dataset and Metadata

In [6]:
display(dataset_df.head())
display(metadata_df.head())

,cell_00,cell_01,cell_02,cell_03,cell_04,cell_05,cell_06,cell_07,cell_08,cell_09,...,cell_16,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,goats_eaten_state,phase_state,value_label
0,2,0,0,2,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.0
1,2,0,0,2,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1.0
2,2,0,2,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1.0
3,0,0,2,0,2,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1.0
4,1,2,0,0,2,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1.0


,episode_id,source_name,source_policy,tiger_opponent,move_index,total_moves,progress_ratio,action,goats_on_board,goats_placed,winner,bucket
0,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,0,20,0.00,80,0,0,Goat,goat_win
1,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,1,20,0.05,105,1,1,Goat,goat_win
2,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,2,20,0.10,75,2,2,Goat,goat_win
3,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,3,20,0.15,0,3,3,Goat,goat_win
4,6,early_robust_goat_vs_normal_tiger,early_robust_goat,normal_tiger,4,20,0.20,45,4,4,Goat,goat_win


### 7. Consistency Checks

In [7]:
expected_state_columns = [f"cell_{idx:02d}" for idx in range(23)] + ["goats_eaten_state", "phase_state"]

checks_df = pd.DataFrame(
    [
        {"check": "npz rows match dataset rows", "passed": len(npz_dataset_df) == len(dataset_df)},
        {"check": "npz rows match metadata rows", "passed": len(npz_dataset_df) == len(metadata_df)},
        {"check": "dataset has expected state columns", "passed": all(column in dataset_df.columns for column in expected_state_columns)},
        {"check": "dataset has value_label", "passed": "value_label" in dataset_df.columns},
        {"check": "npz state column count is 25", "passed": states.shape[1] == 25},
    ]
)
checks_df

,check,passed
0,npz rows match dataset rows,True
1,npz rows match metadata rows,True
2,dataset has expected state columns,True
3,dataset has value_label,True
4,npz state column count is 25,True


### 8. Label and Outcome Distributions

In [8]:
label_counts_df = (
    dataset_df["value_label"]
    .value_counts(dropna=False)
    .rename_axis("value_label")
    .reset_index(name="rows")
    .sort_values("value_label")
)
label_counts_df["ratio"] = label_counts_df["rows"] / len(dataset_df)
display(label_counts_df)

if "bucket" in metadata_df.columns:
    bucket_counts_df = (
        metadata_df["bucket"]
        .value_counts(dropna=False)
        .rename_axis("bucket")
        .reset_index(name="rows")
    )
    bucket_counts_df["ratio"] = bucket_counts_df["rows"] / len(metadata_df)
    display(bucket_counts_df)

,value_label,rows,ratio
1,0.0,8000,0.4
2,0.5,4000,0.2
0,1.0,8000,0.4


,bucket,rows,ratio
0,goat_win,8000,0.4
1,tiger_win,8000,0.4
2,timeout,4000,0.2


### 9. State Feature Summary

In [9]:
dataset_df[expected_state_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
cell_00,20000.0,1.17615,0.560301,0.0,1.0,1.0,2.0,2.0
cell_01,20000.0,0.67435,0.536578,0.0,0.0,1.0,1.0,2.0
cell_02,20000.0,0.72615,0.805350,0.0,0.0,1.0,1.0,2.0
cell_03,20000.0,0.97425,0.988906,0.0,0.0,0.0,2.0,2.0
cell_04,20000.0,0.95255,0.990074,0.0,0.0,0.0,2.0,2.0
cell_05,20000.0,0.67575,0.801089,0.0,0.0,0.0,1.0,2.0
cell_06,20000.0,0.61905,0.530227,0.0,0.0,1.0,1.0,2.0
cell_07,20000.0,0.58875,0.731813,0.0,0.0,0.0,1.0,2.0
cell_08,20000.0,0.77015,0.868307,0.0,0.0,0.0,2.0,2.0
cell_09,20000.0,0.88200,0.560796,0.0,1.0,1.0,1.0,2.0


### 10. Board Cell Value Counts

In [10]:
board_columns = [f"cell_{idx:02d}" for idx in range(23)]

board_value_counts_df = pd.concat(
    [dataset_df[column].value_counts().rename(column) for column in board_columns],
    axis=1,
).fillna(0).astype(int)

board_value_counts_df.index.name = "cell_value"
board_value_counts_df

,cell_00,cell_01,cell_02,cell_03,cell_04,cell_05,cell_06,cell_07,cell_08,cell_09,...,cell_13,cell_14,cell_15,cell_16,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22
cell_value,,,,,,,,,,,,,,,,,,,,,
1,13101,12121,5529,429,351,5063,11475,5907,3865,13432,...,6887,5880,15812,17378,6138,6104,9975,8183,7866,8057
2,5211,683,4497,9528,9350,4226,453,2934,5769,2104,...,824,4024,193,176,1459,501,252,372,310,249
0,1688,7196,9974,10043,10299,10711,8072,11159,10366,4464,...,12289,10096,3995,2446,12403,13395,9773,11445,11824,11694


### 11. Metadata Breakdowns

In [11]:
for column in ["source_policy", "tiger_opponent", "winner", "bucket"]:
    if column in metadata_df.columns:
        display(
            metadata_df[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="rows")
        )

,source_policy,rows
0,stable_normal_goat,4945
1,stable_robust_goat,3912
2,stable_smart_goat,2973
3,mid_smart_goat,1365
4,mid_robust_goat,1261
5,early_normal_goat,1218
6,early_smart_goat,1195
7,early_robust_goat,1071
8,mid_normal_goat,1038
9,random_goat,1022


,tiger_opponent,rows
0,normal_tiger,10693
1,smart_tiger,9307


,winner,rows
0,Goat,8000
1,Tiger,8000
2,MaxTimeout,2578
3,RepeatTimeout,1422


,bucket,rows
0,goat_win,8000
1,tiger_win,8000
2,timeout,4000
